In [1]:
import re
import unicodedata
import pandas as pd

In [2]:
df = pd.read_csv(r"dataset\filtering\news_balanced.csv")
print(f'''
      Shape: {df.shape}
      Columns: {df.columns.tolist()}
''')
df.head()


      Shape: (50, 6)
      Columns: ['content_id', 'text', 'section', 'published_date', 'category', 'keywords']



,content_id,text,section,published_date,category,keywords
0,1442420,"CUPERTINO: Until now, the AirPods Pro were all...",Tech,2024-09-17 00:00:00,Gadgets,"Gadgets,Technology"
1,1572931,A new 100 gigahertz chip that harnesses light ...,Tech,2025-03-07 00:00:00,5G,"SCMP,China,5G,Technology,Internet"
2,1808882,CelcomDigi has refreshed its Postpaid 5G plans...,Tech,2026-02-05 00:00:00,5G,"Telcos,5G,Internet,Technology,Smartphones"
3,1385661,KUALA LUMPUR: The 5G coverage in populated are...,Tech,2024-07-03 00:00:00,5G,"5G,Technology,Internet"
4,1420092,"Apple Inc, seeking new sources of revenue, is ...",Tech,2024-08-15 00:00:00,Gadgets,"Gadgets,Smartphones,Technology,Robotics"


## Text Cleaning

In [3]:
def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    
    text = re.sub(r"''|``|‘’", '"', text)
    
    text = text.replace('\xad', '')
    
    # Links and HTML remove
    text = re.sub(r'\b(?:https?://|www\.)?\S+\.\S+(?:/\S*)?', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    
    # replace control chars with space instead of deleting
    text = ''.join(ch if unicodedata.category(ch)[0] != "C" else ' ' for ch in text)
    
    text = re.sub(r'([.!?])([A-Z])', r'\1 \2', text) # Fix glued sentences
    
    text = re.sub(r'\s+', ' ', text)
    
    text = re.sub(r'©.*?(?:\.|$)', '', text) # Ending year mention remove
    
    text = re.sub(r'\(\s+(?=[A-Za-z–-])', ', ', text) # Open parenthesis to commas
    
    text = re.sub(r'^[A-Z\s/]+(?:,\s*[A-Za-z\s]+)?:\s', '', text) # Location remove
    
    return text.strip()

In [4]:
df['clean_text'] = df['text'].apply(clean_text)

In [5]:
df['published_date'] = pd.to_datetime(df['published_date'])

In [6]:
df.head()

,content_id,text,section,published_date,category,keywords,clean_text
0,1442420,"CUPERTINO: Until now, the AirPods Pro were all...",Tech,2024-09-17,Gadgets,"Gadgets,Technology","Until now, the AirPods Pro were all about keep..."
1,1572931,A new 100 gigahertz chip that harnesses light ...,Tech,2025-03-07,5G,"SCMP,China,5G,Technology,Internet",A new 100 gigahertz chip that harnesses light ...
2,1808882,CelcomDigi has refreshed its Postpaid 5G plans...,Tech,2026-02-05,5G,"Telcos,5G,Internet,Technology,Smartphones",CelcomDigi has refreshed its Postpaid 5G plans...
3,1385661,KUALA LUMPUR: The 5G coverage in populated are...,Tech,2024-07-03,5G,"5G,Technology,Internet",The 5G coverage in populated areas in the coun...
4,1420092,"Apple Inc, seeking new sources of revenue, is ...",Tech,2024-08-15,Gadgets,"Gadgets,Smartphones,Technology,Robotics","Apple Inc, seeking new sources of revenue, is ..."


In [7]:
df['timestamp'] = df['published_date'].astype('int64') // 10**9

In [8]:
df['year'] = df['published_date'].dt.year
df['month'] = df['published_date'].dt.month
df['week'] = df['published_date'].dt.isocalendar().week

In [9]:
import nltk
nltk.download('punkt')
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\gaura\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Sentence Splitting and Naration Fixes

In [10]:
import re

def merge_quote_splits(sentences):
    merged = []
    buffer = ""

    for sent in sentences:
        if buffer:
            buffer += " " + sent
            if buffer.count('"') % 2 == 0:
                merged.append(buffer.strip())
                buffer = ""
        else:
            if sent.count('"') % 2 != 0:
                buffer = sent
            else:
                merged.append(sent)

    if buffer:
        merged.append(buffer)

    return merged

In [11]:
def secondary_split(sent):
    return re.split(r'(?<!")\.\s+(?=[A-Z])', sent)

def refine_sentences(sent_list):
    refined = []
    for s in sent_list:
        refined.extend(secondary_split(s))
    return refined

In [12]:
def refine_sentences(sent_list):
    refined = []
    for s in sent_list:
        refined.extend(secondary_split(s))
    return refined

In [13]:
df['sentences'] = df['clean_text'].apply(
    lambda x: refine_sentences(
        merge_quote_splits(sent_tokenize(x))
    )
)

In [14]:
df_sent = df.explode('sentences').reset_index(drop=True)
df_sent.rename(columns={'sentences': 'sentence'}, inplace=True)

In [16]:
print(df_sent.shape)

category_counts = df_sent['category'].value_counts()
category_counts

(1360, 12)


Gadgets    932
5G         428
Name: category, dtype: int64

In [17]:
df_sent = df_sent[df_sent['sentence'].str.len() > 40]

In [19]:
print(df_sent.shape)

category_counts = df_sent['category'].value_counts()
category_counts

(1291, 12)


Gadgets    883
5G         408
Name: category, dtype: int64

## Removing publisher names from end of articles

In [20]:
boilerplate_pattern= r'[.|"]\s*[–-]\s*[A-Za-z\s]{1,40}$'

df_sent['sentence'] = df_sent['sentence'].str.replace(
    boilerplate_pattern,
    '',
    regex=True
)

In [22]:
df_final = df_sent[['content_id', 'sentence', 'published_date', 'timestamp', 'category']]

print(df_final.shape)
df_final.head()

(1291, 5)


,content_id,sentence,published_date,timestamp,category
0,1442420,"Until now, the AirPods Pro were all about keep...",2024-09-17,1726531200,Gadgets
1,1442420,Apple says the latest AirPods Pro 2 can be use...,2024-09-17,1726531200,Gadgets
2,1442420,"Pending approval by regulatory authorities, th...",2024-09-17,1726531200,Gadgets
3,1442420,The company is also planning to integrate a he...,2024-09-17,1726531200,Gadgets
4,1442420,"If the feature takes hold, Apple could use it ...",2024-09-17,1726531200,Gadgets


In [23]:
df_final.to_csv(r"dataset\preprocessing\prePro-news_balanced.csv", index=False)